# R2 Tool Calling Agent

In [0]:
%pip install --upgrade unitycatalog-langchain[databricks] databricks-langchain langgraph mlflow "langgraph-prebuilt==1.0.8" -q

dbutils.library.restartPython()

In [0]:
import re
import json
from pathlib import Path
import mlflow

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from unitycatalog.ai.core.databricks import DatabricksFunctionClient


LOOKUP_CLAIM = "claims.genai.lookup_claims"

PLAN_AT = "claims.genai.plan_at"

SEARCH_POLICY = "claims.genai.search_policy"

COMPUTE_PAYABLE = "claims.genai.compute_payable"

llm = ChatDatabricks(
    model = "databricks-meta-llama-3-3-70b-instruct",
    temperature = 0,
    max_tokens = 1024
)

uc_client = DatabricksFunctionClient()

In [0]:
toolkit = UCFunctionToolkit(
    function_names = [
        LOOKUP_CLAIM,
        PLAN_AT,
        SEARCH_POLICY,
        COMPUTE_PAYABLE
    ],
    client = uc_client
)

tools = toolkit.tools

In [0]:
import os
system_prompt = """You are Saffron Shield's claims assistant for internal adjuster support.


Route every question by its type, and never mix tool types in one answer:
If the question is about a specific claim's status, facts, or figures, call lookup_claim (and plan_at only if you need the plan terms in force on the admission date). Never call search_policy for this kind of question. State the claim's status exactly as stored, with no rewording.
If the question is about what the policy document says (waiting periods, coverage rules, definitions), call search_policy only. Never call the data tools for this kind of question. Always cite the chunk_id(s) you relied on.
Never guess a number, date, or policy rule from memory. If a tool returns nothing, say so plainly instead of inventing an answer."""
os.makedirs("agent/prompts", exist_ok=True)
with open("agent/prompts/system_v1.md", "w") as f:
    f.write(system_prompt)
print(len(system_prompt.split()), "words")

In [0]:
agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = system_prompt
)

print("Saffron Shield agent created ")

In [0]:
mlflow.set_tracking_uri("databricks")

mlflow.set_experiment(
    "/Shared/Saffron_Shield_R2"
)


mlflow.langchain.autolog()

print("MLflow tracing enabled!")

In [0]:

import csv
import io


def _chunk_ids_from_tool(raw):

    # UC tool results arrive as {"format": "CSV", "value": "<csv rows>"}

    try:

        payload = json.loads(raw) if isinstance(raw, str) else raw

    except (json.JSONDecodeError, TypeError):

        return []

    if not isinstance(payload, dict) or payload.get("format") != "CSV":

        return []

    rows = list(csv.reader(io.StringIO(payload.get("value", ""))))

    if not rows:

        return []

    header = [h.strip() for h in rows[0]]

    if "chunk_id" not in header:

        return []

    col = header.index("chunk_id")

    return [

        r[col] for r in rows[1:]

        if len(r) > col and r[col].strip()

    ]


def extract_chunk_ids(messages, final_answer):

    chunk_ids = []

    for message in messages:

        if getattr(message, "type", "") == "tool":

            raw = message.content

            if not isinstance(raw, str):

                raw = str(raw)

            chunk_ids.extend(_chunk_ids_from_tool(raw))

    # Also catch chunk ids the model cites in its answer

    pattern = (

        r'["\']?chunk_id["\']?\s*[:=]\s*'

        r'["\']?([A-Za-z0-9_\-./#]+)'

    )

    chunk_ids.extend(

        re.findall(

            pattern,

            final_answer,

            flags=re.IGNORECASE

        )

    )

    return list(dict.fromkeys(chunk_ids))

In [0]:

def run_agent(question, output_filename):

    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    messages = result["messages"]

    final_answer = messages[-1].content

    if not isinstance(final_answer, str):
        final_answer = json.dumps(final_answer)

    tool_calls = []

    for message in messages:

        calls = getattr(message, "tool_calls", [])

        for call in calls:

            tool_calls.append({
                "name": call["name"],
                "args": call["args"]
            })

    cited_chunk_ids = extract_chunk_ids(
        messages,
        final_answer
    )

    trace_id = mlflow.get_last_active_trace_id()

    if not trace_id:
        raise RuntimeError(
            "MLflow trace was not generated. "
            "Check MLflow autologging."
        )

    trace_data = {
        "trace_id": trace_id,
        "tool_calls": tool_calls,
        "final_answer": final_answer,
        "cited_chunk_ids": cited_chunk_ids
    }

    output_path = Path('/Workspace/Users/jainanvesha14@gmail.com/Saffron-Shield/genai/agents/agent/traces') / output_filename

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            trace_data,
            file,
            indent=4,
            ensure_ascii=False
        )

    print("Question:", question)
    print("\nFinal answer:", final_answer)
    print("\nTool calls:", tool_calls)
    print("\nMLflow trace ID:", trace_id)
    print("\nJSON saved at:", output_path)

    return trace_data

In [0]:

status_question = "What's the status of claim CLM-10042?"

status_trace = run_agent(
    question=status_question,
    output_filename="r2_status_probe.json"
)

In [0]:

policy_question = (
    "What is the pre-existing disease (PED) "
    "waiting period for a corporate group plan ?"
)

policy_trace = run_agent(
    question=policy_question,
    output_filename="r2_policy_probe.json"
)


# R3 Guardrails

In [0]:
import re
import unicodedata

EMAIL_PATTERN = re.compile(
    r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
    re.IGNORECASE
)

PHONE_PATTERN = re.compile(
    r"(?<!\d)(?:\+91[\s-]?)?[6-9]\d{9}(?!\d)"
)

AADHAAR_PATTERN = re.compile(
    r"(?<!\d)\d{4}[\s-]?\d{4}[\s-]?\d{4}(?!\d)"
)

PAN_PATTERN = re.compile(
    r"\b[A-Z]{5}\d{4}[A-Z]\b",
    re.IGNORECASE
)

PASSPORT_PATTERN = re.compile(
    r"\b[A-Z]\d{7}\b",
    re.IGNORECASE
)


SENSITIVE_REQUESTS = [
    "aadhaar number",
    "aadhar number",
    "pan number",
    "passport number",
    "identity number",
    "identification number",
    "phone number",
    "mobile number",
    "contact number",
    "email address",
    "personal email",
    "bank account number",
    "account number",
    "ifsc code",
    "home address",
    "residential address",
    "contact details",
    "personal details"
]

PAYOUT_PROMISES = [
    "guarantee approval",
    "guarantee payment",
    "guarantee payout",
    "guaranteed approval",
    "guaranteed payout",
    "promise approval",
    "promise payment",
    "promise payout",
    "assure approval",
    "assure payment",
    "confirm guaranteed payment",
    "approve without checking",
    "pay without checking",
    "approve without verification",
    "payout without verification"
]

DOMAIN_KEYWORDS = [
    "claim",
    "claims",
    "insurance",
    "policy",
    "policies",
    "coverage",
    "covered",
    "premium",
    "deductible",
    "copay",
    "co-pay",
    "waiting period",
    "pre-existing",
    "pre existing",
    "ped",
    "hospital",
    "hospitalization",
    "admission",
    "room rent",
    "room limit",
    "network hospital",
    "non-network",
    "non network",
    "member",
    "plan",
    "plans",
    "benefit",
    "benefits",
    "reimbursement",
    "settlement",
    "payable",
    "payout",
    "adjudication",
    "exclusion",
    "exclusions",
    "claim status",
    "claim id",
    "claim number"
]


OUT_OF_SCOPE = [
    "write a poem",
    "write a song",
    "tell me a joke",
    "weather today",
    "stock market",
    "politics",
    "cricket score",
    "football score",
    "recipe",
    "movie recommendation",
    "travel itinerary",
    "solve my homework",
    "write my assignment",
    "debug my code",
    "generate python code",
    "general knowledge"
]


def check_input(text: str) -> dict:

    if not isinstance(text, str):
        return {
            "blocked": True,
            "reason": "Invalid input type"
        }

    normalized = unicodedata.normalize(
        "NFKC", text
    ).strip()

    q = normalized.lower()

    if not q:
        return {
            "blocked": True,
            "reason": "Empty input"
        }

    sensitive_patterns = [
        EMAIL_PATTERN,
        PHONE_PATTERN,
        AADHAAR_PATTERN,
        PAN_PATTERN,
        PASSPORT_PATTERN
    ]

    for pattern in sensitive_patterns:
        if pattern.search(normalized):
            return {
                "blocked": True,
                "reason": "Sensitive identity or contact information detected"
            }

    for keyword in SENSITIVE_REQUESTS:
        if keyword in q:
            return {
                "blocked": True,
                "reason": "Request for sensitive personal information"
            }

    for phrase in PAYOUT_PROMISES:
        if phrase in q:
            return {
                "blocked": True,
                "reason": "Unsupported payout or approval promise"
            }

    for phrase in OUT_OF_SCOPE:
        if phrase in q:
            return {
                "blocked": True,
                "reason": "Question is outside Saffron Shield scope"
            }

    if not any(keyword in q for keyword in DOMAIN_KEYWORDS):
        return {
            "blocked": True,
            "reason": "Question is outside Saffron Shield claims and policies"
        }

    return {
        "blocked": False,
        "reason": "Input passed guardrails"
    }


In [0]:


test_prompts = [

    "What's the status of claim CLM-10042?",

    "What is the PED waiting period for a corporate group plan?",

    "What is the coverage limit under my insurance policy?",

    "Tell me the Aadhaar number of the claimant",

    "Give me the claimant's phone number",

    "Write a poem about cricket",

    "Guarantee approval of my insurance claim",

    "Send the claimant's email address",

    "How is the payable amount calculated for a claim?",

    "What are the exclusions in the policy?"
]


for question in test_prompts:

    result = check_input(question)

    print("Question:", question)
    print("Result:", result)
    print("-" * 60)


In [0]:

def run_safe_agent(question):

    # STEP 1: Check input BEFORE invoking the agent

    guardrail_result = check_input(question)

    if guardrail_result["blocked"]:

        return {
            "answer": (
                "I can only assist with Saffron Shield "
                "claims and policy-related questions. "
                "I cannot provide sensitive personal "
                "information or guarantee claim payouts."
            ),
            "blocked": True,
            "stage": "input",
            "tool_calls": []
        }

    # STEP 2: Only allowed inputs reach the agent

    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    messages = result["messages"]

    answer = messages[-1].content

    # STEP 3: Extract actual tool calls

    tool_calls = []

    for message in messages:

        for call in getattr(message, "tool_calls", []):

            tool_calls.append({
                "name": call["name"],
                "args": call["args"]
            })

    # STEP 4: Return result

    return {
        "answer": answer,
        "blocked": False,
        "stage": None,
        "tool_calls": tool_calls
    }


In [0]:

def check_output(text: str) -> dict:

    if not isinstance(text, str):
        return {"blocked": True}

    patterns = [
        EMAIL_PATTERN,
        PHONE_PATTERN,
        AADHAAR_PATTERN,
        PAN_PATTERN,
        PASSPORT_PATTERN
    ]

    for pattern in patterns:
        if pattern.search(text):
            return {"blocked": True}

    return {"blocked": False}


In [0]:
# output_check = check_output(answer)

# if output_check["blocked"]:

#     return {
#         "answer": (
#             "I cannot display this response because "
#             "it may contain sensitive personal information."
#         ),
#         "blocked": True,
#         "stage": "output",
#         "tool_calls": tool_calls
#     }

In [0]:

import json
import time
from pathlib import Path

# from src.guardrails import check_input

PROBE_FILE = Path("/Volumes/claims/raw/landing/guardrail_probes.json")

OUTPUT_FILE = Path("/Workspace/Users/jainanvesha14@gmail.com/Saffron-Shield/genai/agents/agent/traces/r3_guardrails.json")

# Load client probes
with open(PROBE_FILE, "r", encoding="utf-8") as f:
    probes = json.load(f)["probes"]

results = []

for probe in probes:

    question = probe["prompt"]

    check = check_input(question)

    if check["blocked"]:

        result = {
            "id": probe["id"],
            "decision": "blocked",
            "stage": "input",
            "answer": (
                "I can only assist with Saffron Shield "
                "claims and policies. I cannot provide "
                "sensitive personal information or "
                "guarantee claim payouts."
            ),
            "tool_calls": []
        }

    else:

        # The model can intermittently return an empty response that the
        # endpoint rejects as "Model response did not respect the required
        # format". Retry, as the error message recommends.

        execution = None
        last_error = None

        for attempt in range(3):
            try:
                execution = run_safe_agent(question)
                break
            except Exception as e:
                last_error = e
                time.sleep(2)

        if execution is None:
            raise last_error

        result = {
            "id": probe["id"],
            "decision": "allowed",
            "stage": None,
            "answer": execution["answer"],
            "tool_calls": execution["tool_calls"]
        }

    results.append(result)


# Create evidence
evidence = {
    "ai_gateway": (
        "Not used: deterministic input guardrails "
        "implemented using Python standard library."
    ),
    "results": results
}

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        evidence,
        f,
        indent=4,
        ensure_ascii=False
    )

print("R3 evidence generated:", OUTPUT_FILE)


# R4 Short and Long term memory 

In [0]:

%pip install --upgrade "databricks-sdk>=0.89.0" "psycopg[binary]>=3.1.0" -q

In [0]:

from databricks.sdk import WorkspaceClient
import psycopg

# Databricks workspace client
workspace_client = WorkspaceClient()

# Lakebase connection details
LAKEBASE_ENDPOINT = (
    "projects/saffron-shiled/branches/production/endpoints/primary"
)

LAKEBASE_HOST = "ep-icy-frost-d8oosse9.database.us-east-2.cloud.databricks.com"
LAKEBASE_PORT = 5432
LAKEBASE_DATABASE = "databricks_postgres"

# Use the PostgreSQL OAuth role created for your Databricks identity.
# For interactive user authentication, this is normally your email.
LAKEBASE_USER = "jainanvesha14@gmail.com"

def get_lakebase_connection():

    # Generate a fresh OAuth database credential
    credential = workspace_client.postgres.generate_database_credential(
        endpoint=LAKEBASE_ENDPOINT
    )

    # Connect to Lakebase PostgreSQL
    connection = psycopg.connect(
        host=LAKEBASE_HOST,
        port=LAKEBASE_PORT,
        dbname=LAKEBASE_DATABASE,
        user=LAKEBASE_USER,
        password=credential.token,
        sslmode="require",
        connect_timeout=15
    )

    return connection

print("Lakebase connection configuration created.")


In [0]:
connection = get_lakebase_connection()

try:
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT current_user, current_database(), version()"
        )

        result = cursor.fetchone()

        print("Connected successfully!")
        print("Database user:", result[0])
        print("Database:", result[1])
        print("PostgreSQL:", result[2][:80])

finally:
    connection.close()


In [0]:

connection = get_lakebase_connection()

try:
    with connection.cursor() as cursor:

        cursor.execute("""
            CREATE SCHEMA IF NOT EXISTS genai;
        """)

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS genai.adjuster_notes (
                note_id BIGSERIAL PRIMARY KEY,
                user_id TEXT NOT NULL,
                note TEXT NOT NULL,
                saved_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
            );
        """)

    connection.commit()

    print("Lakebase memory table created successfully.")

finally:
    connection.close()


In [0]:

import json
from pathlib import Path

MEMORY_PROBE_PATH = Path("/Volumes/claims/raw/landing/memory_probes.json")

with open(MEMORY_PROBE_PATH, "r", encoding="utf-8") as file:
    memory_probe_data = json.load(file)

USER_ID = memory_probe_data["user_id"]

print("Adjuster ID:", USER_ID)


In [0]:

from langchain.tools import tool


def create_memory_tools(user_id):

    @tool
    def save_note(note: str) -> str:
        """
        Save an important claim-related note for the current adjuster.
        Use this when the adjuster explicitly asks to save or remember
        information for a future conversation.
        """

        if not note or not note.strip():
            return "Cannot save an empty note."

        connection = get_lakebase_connection()

        try:
            with connection.cursor() as cursor:

                cursor.execute(
                    """
                    INSERT INTO genai.adjuster_notes
                        (user_id, note)
                    VALUES (%s, %s)
                    """,
                    (user_id, note.strip())
                )

            connection.commit()

            return "Note successfully saved in Lakebase."

        finally:
            connection.close()


    @tool
    def recall_notes() -> str:
        """
        Retrieve previously saved notes for the current adjuster.
        Use this when the adjuster refers to a previous conversation
        or asks about information they previously asked you to remember.
        """

        connection = get_lakebase_connection()

        try:
            with connection.cursor() as cursor:

                cursor.execute(
                    """
                    SELECT note, saved_at
                    FROM genai.adjuster_notes
                    WHERE user_id = %s
                    ORDER BY saved_at DESC
                    LIMIT 20
                    """,
                    (user_id,)
                )

                rows = cursor.fetchall()

            if not rows:
                return "No saved notes found for this adjuster."

            notes = [
                f"- {row[0]}" for row in rows
            ]

            return "Previously saved adjuster notes:\n" + "\n".join(notes)

        finally:
            connection.close()


    return [save_note, recall_notes]


memory_tools = create_memory_tools(USER_ID)

print("Memory tools:", [tool.name for tool in memory_tools])


In [0]:
tools = toolkit.tools

In [0]:

all_tools = tools + memory_tools

print("R2 tools:", len(tools))
print("R4 memory tools:", len(memory_tools))
print("Total tools:", len(all_tools))


In [0]:

memory_instructions = """

ADJUSTER MEMORY RULES:

1. When an adjuster explicitly asks you to save or remember
   a claim-related note for a future conversation, call save_note.

2. When an adjuster asks about previously saved information
   or refers to an earlier conversation, call recall_notes.

3. Do not claim that a note was saved unless save_note
   successfully returns confirmation.

4. Use recalled notes as additional context. Do not treat
   them as a replacement for current claim facts or policy documents.

5. Never invent saved notes. If recall_notes finds nothing,
   tell the adjuster that no saved note was found.

6. Use the existing R2 tool-selection rules for claim facts,
   plan information, policy questions, and payout calculations.
"""

r4_system_prompt = system_prompt + memory_instructions

print("R4 system prompt created.")


In [0]:

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

# Short-term conversation memory
short_term_memory = InMemorySaver()

memory_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=r4_system_prompt,
    checkpointer=short_term_memory
)

print("R4 agent created successfully.")


# R5 Supervisor Agent 

In [0]:
R5_QUESTION = (
    "For CLM-20003: which plan applied, is room rent over cap, "
    "and what is payable ?"
)

R5_CLAIM_ID = "CLM-20003"

R5_TRACE_DIR = Path("agent/traces")
R5_TRACE_DIR.mkdir(parents=True, exist_ok=True)

In [0]:
# Existing R2 variable: tools
# Each tool should expose its UC function name through .name.

tool_map = {}

for tool in tools:
    tool_name = getattr(tool, "name", "")
    short_name = tool_name.split("__")[-1].upper()

    if short_name in {
        "LOOKUP_CLAIMS",
        "PLAN_AT",
        "SEARCH_POLICY",
        "COMPUTE_PAYABLE"
    }:
        tool_map[short_name] = tool

required_tools = {
    "LOOKUP_CLAIMS",
    "PLAN_AT",
    "SEARCH_POLICY",
    "COMPUTE_PAYABLE"
}

missing_tools = required_tools - set(tool_map.keys())

if missing_tools:
    raise ValueError(f"Missing R1 tools: {missing_tools}")

print("R5 tools ready:")
for name in sorted(tool_map):
    print("-", name)

In [0]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import SystemMessage
from typing import Any
import json


class ToolCallLimitMiddleware(AgentMiddleware):
    """Cap the number of tool calls an agent can make.

    Once the cap is reached the model is told to answer directly; if it
    still tries to call a tool, the pending call is cancelled so the
    agent ends with the best available answer.
    """

    def __init__(self, max_tool_calls: int):
        super().__init__()
        self.max_tool_calls = max_tool_calls

    def _tool_calls_done(self, state) -> int:
        return sum(
            1 for m in state["messages"]
            if getattr(m, "type", "") == "tool"
        )

    def before_model(self, state, runtime) -> dict[str, Any] | None:
        if self._tool_calls_done(state) >= self.max_tool_calls:
            return {"messages": [SystemMessage(
                "You have reached the tool call limit. Do not call any "
                "more tools. Write your final answer now based on the "
                "tool results you already have, including the required "
                "JSON verdict."
            )]}
        return None

    def after_model(self, state, runtime) -> dict[str, Any] | None:
        messages = state["messages"]
        last = messages[-1]

        if not getattr(last, "tool_calls", None):
            return None
        if self._tool_calls_done(state) < self.max_tool_calls:
            return None

        # Hard stop: cancel the pending tool call(s).
        last.tool_calls = []
        last.additional_kwargs.pop("tool_calls", None)

        if not last.content:
            inner = None
            for m in messages:
                if getattr(m, "type", "") == "tool":
                    inner = m.content
            # Unwrap {"format": ..., "value": ...} tool results.
            try:
                payload = json.loads(inner) if isinstance(inner, str) else inner
                if isinstance(payload, dict) and "value" in payload:
                    inner = payload["value"]
                inner = json.loads(inner) if isinstance(inner, str) else inner
            except (json.JSONDecodeError, TypeError):
                pass
            last.content = (
                "Tool call limit reached. Returning the tool verdict "
                "directly: " + json.dumps({"verdict": inner}, default=str)
            )
        return None


data_tools = [
    tool_map["LOOKUP_CLAIMS"],
    tool_map["PLAN_AT"]
]

policy_tools = [
    tool_map["SEARCH_POLICY"]
]

adjudication_tools = [
    tool_map["COMPUTE_PAYABLE"]
]

data_agent = create_agent(
    model=llm,
    tools=data_tools,
    system_prompt="""
You are the Data Specialist for Saffron Shield.

Your responsibility is to retrieve factual claim and plan information.

Rules:
1. Use lookup_claim to retrieve the facts of the claim.
2. Use plan_at to identify the plan applicable at admission.
3. Do not calculate or decide the payable amount.
4. Do not invent missing claim or plan information.
5. Return the retrieved facts clearly, including the tool results.
6. Your final answer MUST report the exact values of these fields,
   copied verbatim from the tool results:
   claim_id, member_id, billed_amount, is_pre_existing,
   plan_code_at_admission, plan_family, room_rent_per_day,
   room_rent_cap_per_day, room_rent_over_cap, policy_start_date,
   policy_tenure_months, days_since_policy_start,
   is_network_at_admission.
"""
)

policy_agent = create_agent(
    model=llm,
    tools=policy_tools,
    system_prompt="""
You are the Policy Specialist for Saffron Shield.

Your responsibility is to find the policy rules concerning room rent
capping and proportionate deduction.

Rules:
1. Search the policy using search_policy.
2. Find the relevant section in
   04_Room_Rent_Capping_and_Proportionate_Deduction.md.
3. Return the relevant policy text.
4. Include the exact chunk_id values returned by the search tool.
5. Do not decide the claim payout.
6. Do not invent policy clauses or chunk IDs.
"""
)

adjudication_agent = create_agent(
    model=llm,
    tools=adjudication_tools,
    middleware=[ToolCallLimitMiddleware(max_tool_calls=2)],
    system_prompt="""
You are the Adjudication Specialist for Saffron Shield.

You receive:
- Claim and plan facts from the Data Specialist.
- Room-rent policy evidence from the Policy Specialist.

Your responsibility is to call compute_payable using the actual
claim and plan facts.

Rules:
1. You MUST call compute_payable.
2. Do not calculate a payout using your own reasoning.
3. Do not invent input values.
4. Use the plan applicable at admission.
5. Return the exact compute_payable input arguments and its result.
6. Clearly return a JSON object with a verdict containing:
   plan_code, payable, waiting_period_ok, network_at_admission,
   room_rent_over_cap, room_rent_deduction_pct, copay_pct,
   reason_code and payable_amount.
7. If required information is missing, explicitly report it.
"""
)

print("Three specialist agents created using create_agent.")

In [0]:
def run_specialist(agent, question):
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    messages = result["messages"]

    final_answer = messages[-1].content

    if not isinstance(final_answer, str):
        final_answer = json.dumps(final_answer, default=str)

    tool_calls = []
    tool_outputs = []

    for message in messages:
        # Capture actual tool calls
        for call in getattr(message, "tool_calls", []) or []:
            tool_calls.append(call.get("name"))

        # Capture tool results
        if getattr(message, "type", "") == "tool":
            tool_outputs.append({
                "tool": getattr(message, "name", ""),
                "content": str(message.content)
            })

    return {
        "answer": final_answer,
        "tool_calls": tool_calls,
        "tool_outputs": tool_outputs,
        "messages": messages
    }

In [0]:
import uuid
def run_r5_supervisor(question):

    trace_id = str(uuid.uuid4())

    # STEP 1: DATA SPECIALIST
    data_result = run_specialist(
        data_agent,
        f"""
        Retrieve all necessary claim and plan facts for claim {R5_CLAIM_ID}.

        Identify the plan applicable at admission.
        Use lookup_claim and plan_at.
        Do not calculate the payout.

        Original client question:
        {question}
        """
    )

    # STEP 2: POLICY SPECIALIST
    policy_result = run_specialist(
        policy_agent,
        f"""
        Find the room-rent capping and proportionate deduction
        policy applicable to the following question.

        Claim: {R5_CLAIM_ID}

        Original question:
        {question}

        Search the document:
        04_Room_Rent_Capping_and_Proportionate_Deduction.md

        Return the relevant text and exact chunk IDs.
        """
    )

    # STEP 3: ADJUDICATION SPECIALIST
    adjudication_result = run_specialist(
        adjudication_agent,
        f"""
        Determine the claim adjudication by calling compute_payable.

        ORIGINAL QUESTION:
        {question}

        DATA SPECIALIST RESULTS:
        {data_result["answer"]}

        POLICY SPECIALIST RESULTS:
        {policy_result["answer"]}

        Use only the actual facts and policy evidence supplied.
        Call compute_payable and return its exact input arguments
        and result.
        """
    )

    return {
        "trace_id": trace_id,
        "data": data_result,
        "policy": policy_result,
        "adjudication": adjudication_result
    }

In [0]:
def extract_compute_payable_call(adjudication_result):

    for message in adjudication_result["messages"]:
        for call in getattr(message, "tool_calls", []) or []:

            # Tool call names arrive as 'claims__genai__compute_payable'.
            short_name = (
                call.get("name", "")
                .split("__")[-1]
                .split(".")[-1]
                .lower()
            )

            if short_name == "compute_payable":
                return call.get("args", {})

    raise ValueError(
        "Adjudication agent did not call compute_payable."
    )

In [0]:
def parse_json_result(raw):

    # Handle LangChain tool message objects
    if hasattr(raw, "content"):
        raw = raw.content

    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except json.JSONDecodeError:
            raise ValueError(
                f"Could not parse compute_payable result: {raw}"
            )

    if not isinstance(raw, dict):
        raise ValueError(
            f"Could not parse compute_payable result: {raw}"
        )

    # Unwrap {"format": "SCALAR", "value": "<json string>"} results.
    if "format" in raw and "value" in raw:
        inner = raw["value"]
        if isinstance(inner, str):
            try:
                inner = json.loads(inner)
            except json.JSONDecodeError:
                inner = None
        if isinstance(inner, dict):
            return inner

    return raw


def extract_verdict_json(adjudication_text):
    """Find the JSON object in the adjudication answer that carries
    the verdict. Tolerates extra JSON blocks such as the exact
    compute_payable input arguments."""

    candidates = []
    depth = 0
    start = None

    for i, ch in enumerate(adjudication_text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            if depth:
                depth -= 1
                if depth == 0 and start is not None:
                    candidates.append(
                        adjudication_text[start:i + 1]
                    )

    for candidate in candidates:
        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if not isinstance(payload, dict):
            continue
        if isinstance(payload.get("verdict"), dict):
            return payload["verdict"]
        if "payable_amount" in payload:
            return payload

    raise ValueError(
        "Adjudication response must contain a JSON object "
        "with its verdict."
    )


def run_verifier(compute_inputs, adjudication_result):

    # Re-run the deterministic UC function.
    recomputed_raw = tool_map["COMPUTE_PAYABLE"].invoke(
        compute_inputs
    )

    recomputed = parse_json_result(recomputed_raw)

    # Parse the adjudication specialist's response.
    adjudication_text = adjudication_result["answer"]

    adjudicated_verdict = extract_verdict_json(
        adjudication_text
    )

    # Compare all required verdict fields.
    verdict_fields = [
        "plan_code",
        "payable",
        "waiting_period_ok",
        "network_at_admission",
        "room_rent_over_cap",
        "room_rent_deduction_pct",
        "copay_pct",
        "reason_code",
        "payable_amount"
    ]

    # compute_payable names the reason code "deduction_code".
    source_key = {
        field: ("deduction_code" if field == "reason_code" else field)
        for field in verdict_fields
    }

    verified_verdict = {
        field: recomputed[source_key[field]]
        for field in verdict_fields
    }

    agrees = all(
        adjudicated_verdict.get(
            field, adjudicated_verdict.get(source_key[field])
        ) == verified_verdict[field]
        for field in verdict_fields
    )

    return {
        "agrees": agrees,
        "verdict": verified_verdict,
        "recomputed_result": recomputed
    }

In [0]:
import csv
import io


def extract_chunk_ids(policy_result):

    chunk_ids = []

    for output in policy_result["tool_outputs"]:
        content = output["content"]

        # search_policy results arrive as
        # {"format": "CSV", "value": "<csv rows>"} where the first
        # column of each row is the chunk_id.
        try:
            payload = (
                json.loads(content)
                if isinstance(content, str)
                else content
            )
        except (json.JSONDecodeError, TypeError):
            continue

        if not (
            isinstance(payload, dict)
            and payload.get("format") == "CSV"
        ):
            continue

        rows = list(
            csv.reader(io.StringIO(payload.get("value", "")))
        )

        for row in rows[1:]:
            if row and row[0].strip():
                chunk_ids.append(row[0].strip())

    # Remove duplicates while preserving order.
    return list(dict.fromkeys(chunk_ids))

In [0]:
def create_r5_final_answer(data_result, policy_result, verified_verdict):

    v = verified_verdict

    answer = (
        f"For claim {R5_CLAIM_ID}, the plan applicable at admission "
        f"was {v['plan_code']}. "
    )

    if v["room_rent_over_cap"]:
        answer += (
            f"The room rent was over the applicable cap. "
            f"The room-rent proportionate deduction is "
            f"{v['room_rent_deduction_pct']}%. "
        )
    else:
        answer += "The room rent was not over the applicable cap. "

    answer += (
        f"The verified payable amount is "
        f"{v['payable_amount']}. "
        f"Payable status: {v['payable']}. "
        f"Reason code: {v['reason_code']}."
    )

    return answer

In [0]:
def run_r5_attempt(attempt):

    if attempt not in [1, 2, 3]:
        raise ValueError("R5 supports at most three attempts.")

    print(f"Running R5 attempt {attempt}...")

    # Run the supervisor and specialists.
    results = run_r5_supervisor(R5_QUESTION)

    # Get the exact compute_payable arguments.
    compute_inputs = extract_compute_payable_call(
        results["adjudication"]
    )

    # Run the plain Python verifier.
    verifier_result = run_verifier(
        compute_inputs,
        results["adjudication"]
    )

    # Extract policy citations.
    cited_chunk_ids = extract_chunk_ids(
        results["policy"]
    )

    # Create final answer from verified output.
    final_answer = create_r5_final_answer(
        results["data"],
        results["policy"],
        verifier_result["verdict"]
    )

    # Construct evidence in the required format.
    evidence = {
        "question": R5_QUESTION,
        "attempt": attempt,
        "trace_id": results["trace_id"],

        "route": [
            "data",
            "policy",
            "adjudication",
            "verifier"
        ],

        "specialists": {
            "data": {
                "tool_calls": results["data"]["tool_calls"]
            },

            "policy": {
                "tool_calls": results["policy"]["tool_calls"],
                "cited_chunk_ids": cited_chunk_ids
            },

            "adjudication": {
                "tool_calls": results["adjudication"]["tool_calls"]
            }
        },

        "verifier": {
            "agrees": verifier_result["agrees"]
        },

        "verdict": verifier_result["verdict"],

        "final_answer": final_answer
    }

    # Save each attempt separately. Never overwrite an earlier attempt.
    output_file = (
        R5_TRACE_DIR / f"r5_attempt_{attempt}.json"
    )

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(evidence, f, indent=2, ensure_ascii=False)

    print(f"Evidence saved: {output_file}")
    print("Verifier agrees:", verifier_result["agrees"])
    print("\nFinal answer:")
    print(final_answer)

    return evidence

In [0]:
r5_attempt_1 = run_r5_attempt(1)

In [0]:
def short_tool_name(name):
    # Tool call names arrive as 'claims__genai__lookup_claims'.
    return str(name).split("__")[-1].split(".")[-1].lower()


def validate_r5_evidence(evidence):

    specialists = evidence["specialists"]

    assert evidence["question"] == R5_QUESTION

    assert "lookup_claims" in [
        short_tool_name(name)
        for name in specialists["data"]["tool_calls"]
    ], "Data agent did not call lookup_claim."

    assert "compute_payable" in [
        short_tool_name(name)
        for name in specialists["adjudication"]["tool_calls"]
    ], "Adjudication agent did not call compute_payable."

    assert specialists["policy"]["cited_chunk_ids"], (
        "Policy agent did not produce any chunk IDs."
    )

    assert evidence["route"] == [
        "data",
        "policy",
        "adjudication",
        "verifier"
    ]

    assert evidence["verifier"]["agrees"] is True, (
        "Verifier disagrees with the adjudication verdict."
    )

    assert evidence["verdict"]["plan_code"], (
        "Plan code is missing."
    )

    print("R5 evidence validation passed.")

# R6 MCP

In [0]:
%pip install --upgrade databricks-mcp nest_asyncio -q

dbutils.library.restartPython()

In [0]:
import nest_asyncio

nest_asyncio.apply()

import json
from pathlib import Path

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

# Managed MCP server exposing the claims.genai UC functions.
workspace_client = WorkspaceClient()

MCP_SERVER_URL = (
    f"{workspace_client.config.host}"
    "/api/2.0/mcp/functions/claims/genai"
)

mcp_client = DatabricksMCPClient(
    server_url=MCP_SERVER_URL,
    workspace_client=workspace_client
)

mcp_tools = mcp_client.list_tools()

tool_names = sorted(tool.name for tool in mcp_tools)

print("Transport: managed")
print("MCP server:", MCP_SERVER_URL)
print("Tools listed by the MCP server:")
for name in tool_names:
    print(" -", name)

# MCP data call: one of the Paper 4 probe claims.
lookup_result = mcp_client.call_tool(
    "claims__genai__lookup_claims",
    {"claim_id": "CLM-20003"}
)

lookup_text = lookup_result.content[0].text

# Build the payout inputs from the claim facts returned via MCP.
lookup_payload = json.loads(lookup_text)

claim_row = dict(
    zip(lookup_payload["columns"], lookup_payload["rows"][0])
)

payout_inputs = {
    "plan_code": claim_row["plan_code_at_admission"],
    "is_pre_existing": claim_row["is_pre_existing"],
    "policy_tenure_months": claim_row["policy_tenure_months"],
    "days_since_policy_start": claim_row["days_since_policy_start"],
    "is_network_at_admission": claim_row["is_network_at_admission"],
    "room_rent_per_day": claim_row["room_rent_per_day"],
    "billed_amount": claim_row["billed_amount"]
}

# MCP payout call for the same claim.
compute_result = mcp_client.call_tool(
    "claims__genai__compute_payable",
    payout_inputs
)

compute_text = compute_result.content[0].text

mcp_verdict = json.loads(
    json.loads(compute_text)["rows"][0][0]
)

# The verified answer from the R5 plain-code verifier.
R5_EVIDENCE_PATH = Path(
    "/Workspace/Users/jainanvesha14@gmail.com/Saffron-Shield"
    "/genai/agents/agent/traces/r5_attempt_1.json"
)

with open(R5_EVIDENCE_PATH, "r", encoding="utf-8") as file:
    r5_evidence = json.load(file)

verified_answer = r5_evidence["verdict"]

# Compare all verdict fields; compute_payable names the reason
# code "deduction_code" while the verified verdict uses
# "reason_code" (same mapping as the R5 verifier).
verdict_fields = [
    "plan_code",
    "payable",
    "waiting_period_ok",
    "network_at_admission",
    "room_rent_over_cap",
    "room_rent_deduction_pct",
    "copay_pct",
    "reason_code",
    "payable_amount"
]

payout_matches_verified = all(
    mcp_verdict.get(
        field,
        mcp_verdict.get(
            "deduction_code" if field == "reason_code" else field
        )
    ) == verified_answer[field]
    for field in verdict_fields
)

print("\nlookup_claims result:")
print(lookup_text)
print("\ncompute_payable result:")
print(compute_text)
print("\nPayout equals the verified R5 answer:",
      payout_matches_verified)

assert payout_matches_verified, (
    "MCP payout does not equal the verified answer."
)

# Deliverable: evidence/r6_mcp.json
Path("evidence").mkdir(exist_ok=True)

r6_mcp_evidence = {
    "transport": "managed",
    "server": MCP_SERVER_URL,
    "tools": tool_names,
    "calls": [
        {
            "tool": "claims__genai__lookup_claims",
            "arguments": {"claim_id": "CLM-20003"},
            "result": lookup_text
        },
        {
            "tool": "claims__genai__compute_payable",
            "arguments": payout_inputs,
            "result": compute_text
        }
    ]
}

evidence_path = Path("evidence") / "r6_mcp.json"

with open(evidence_path, "w", encoding="utf-8") as file:
    json.dump(
        r6_mcp_evidence,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nJSON saved at:", evidence_path)

In [0]:
import csv
import io
import json
import re
from pathlib import Path

import mlflow

from langchain.agents import create_agent
from langchain_core.tools import StructuredTool
from databricks_langchain import ChatDatabricks
from pydantic import create_model

mlflow.set_tracking_uri("databricks")

mlflow.set_experiment("/Shared/Saffron_Shield_R2")

mlflow.langchain.autolog()

llm = ChatDatabricks(
    model="databricks-meta-llama-3-3-70b-instruct",
    temperature=0,
    max_tokens=1024
)

system_prompt = """You are Saffron Shield's claims assistant for internal adjuster support.


Route every question by its type, and never mix tool types in one answer:
If the question is about a specific claim's status, facts, or figures, call lookup_claim (and plan_at only if you need the plan terms in force on the admission date). Never call search_policy for this kind of question. State the claim's status exactly as stored, with no rewording.
If the question is about what the policy document says (waiting periods, coverage rules, definitions), call search_policy only. Never call the data tools for this kind of question. Always cite the chunk_id(s) you relied on.
Never guess a number, date, or policy rule from memory. If a tool returns nothing, say so plainly instead of inventing an answer."""

# Take the tools from the MCP server and expose them to the agent.
JSON_TO_PY = {
    "string": str,
    "number": float,
    "integer": int,
    "boolean": bool
}


def mcp_tool_to_langchain(mcp_tool):

    fields = {}

    for pname, pschema in (
        mcp_tool.inputSchema.get("properties", {}).items()
    ):
        pytype = JSON_TO_PY.get(pschema.get("type", "string"), str)
        fields[pname] = (pytype, ...)

    args_model = create_model(mcp_tool.name + "_args", **fields)

    def call_mcp_tool(**kwargs):
        return mcp_client.call_tool(
            mcp_tool.name,
            kwargs
        ).content[0].text

    return StructuredTool(
        name=mcp_tool.name,
        description=mcp_tool.description,
        args_schema=args_model,
        func=call_mcp_tool
    )


mcp_langchain_tools = [
    mcp_tool_to_langchain(mcp_tool)
    for mcp_tool in mcp_tools
]

print("Agent tools taken from the MCP server:")
for langchain_tool in mcp_langchain_tools:
    print(" -", langchain_tool.name)

mcp_agent = create_agent(
    model=llm,
    tools=mcp_langchain_tools,
    system_prompt=system_prompt
)

print("\nMCP agent created.")

R6_QUESTION = (
    "For CLM-20003: which plan applied, is room rent over cap, "
    "and what is payable ?"
)

result = mcp_agent.invoke({
    "messages": [
        {"role": "user", "content": R6_QUESTION}
    ]
})

messages = result["messages"]

final_answer = messages[-1].content

if not isinstance(final_answer, str):
    final_answer = json.dumps(final_answer)

tool_calls = []

for message in messages:

    for call in getattr(message, "tool_calls", []) or []:

        tool_calls.append({
            "name": call["name"],
            "args": call["args"]
        })


def _chunk_ids_from_tool(raw):

    # MCP tool results arrive as a JSON table:
    # {"columns": [...], "rows": [[...], ...]}

    try:

        payload = json.loads(raw) if isinstance(raw, str) else raw

    except (json.JSONDecodeError, TypeError):

        return []

    if not isinstance(payload, dict) or "columns" not in payload:

        return []

    header = [str(h).strip() for h in payload["columns"]]

    if "chunk_id" not in header:

        return []

    col = header.index("chunk_id")

    return [
        str(r[col]) for r in payload.get("rows", [])
        if len(r) > col and str(r[col]).strip()
    ]


def extract_chunk_ids(messages, final_answer):

    chunk_ids = []

    for message in messages:

        if getattr(message, "type", "") == "tool":

            raw = message.content

            if not isinstance(raw, str):

                raw = str(raw)

            chunk_ids.extend(_chunk_ids_from_tool(raw))

    # Also catch chunk ids the model cites in its answer

    pattern = (
        r'["\']?chunk_id["\']?\s*[:=]\s*'
        r'["\']?([A-Za-z0-9_\-./#]+)'
    )

    chunk_ids.extend(
        re.findall(
            pattern,
            final_answer,
            flags=re.IGNORECASE
        )
    )

    return list(dict.fromkeys(chunk_ids))


cited_chunk_ids = extract_chunk_ids(messages, final_answer)

trace_id = mlflow.get_last_active_trace_id()

if not trace_id:
    raise RuntimeError(
        "MLflow trace was not generated. "
        "Check MLflow autologging."
    )

trace_data = {
    "trace_id": trace_id,
    "tool_calls": tool_calls,
    "final_answer": final_answer,
    "cited_chunk_ids": cited_chunk_ids
}

output_path = Path(
    "/Workspace/Users/jainanvesha14@gmail.com/Saffron-Shield"
    "/genai/agents/agent/traces/r6_mcp_agent.json"
)

with open(output_path, "w", encoding="utf-8") as file:

    json.dump(
        trace_data,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nQuestion:", R6_QUESTION)
print("\nFinal answer:", final_answer)
print("\nTool calls:")
for call in tool_calls:
    print("  ", call["name"], call["args"])
print("\nMLflow trace ID:", trace_id)
print("\nJSON saved at:", output_path)

# R7 Log, register, version, serve

In [0]:
import hashlib
from pathlib import Path

AGENTS_DIR = Path(
    "/Workspace/Users/jainanvesha14@gmail.com/Saffron-Shield"
    "/genai/agents"
)
PROMPTS_DIR = AGENTS_DIR / "agent" / "prompts"

def prompt_sha(path):
    """First 12 hex characters of the SHA-256 of the prompt file
    text, leading and trailing whitespace removed."""
    text = path.read_text(encoding="utf-8").strip()
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

v1_text = (PROMPTS_DIR / "system_v1.md").read_text(encoding="utf-8").strip()
v2_text = (PROMPTS_DIR / "system_v2.md").read_text(encoding="utf-8").strip()

v1_sha = prompt_sha(PROMPTS_DIR / "system_v1.md")
v2_sha = prompt_sha(PROMPTS_DIR / "system_v2.md")

# Deliverable checks: the v2 prompt is at least 40 words and differs
# from the v1 prompt.
assert len(v2_text.split()) >= 40, "v2 prompt must be at least 40 words"
assert v1_text != v2_text, "v2 prompt must differ from the v1 prompt"

print("v1 prompt:", len(v1_text.split()), "words | sha:", v1_sha)
print("v2 prompt:", len(v2_text.split()), "words | sha:", v2_sha)

In [0]:
import time

import mlflow
from mlflow import MlflowClient
from mlflow.models import ModelSignature
from mlflow.models.resources import DatabricksServingEndpoint
from mlflow.types.schema import ColSpec, Schema

REGISTERED_MODEL = "claims.genai.claims_agent"
AGENT_CODE = AGENTS_DIR / "r7_agent_code.py"

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/Saffron_Shield_R2")

import pandas as pd

# Only mlflow + pandas: enough for this pyfunc agent, and light enough
# for Model Serving express deployment (fast container build).
PIP_REQUIREMENTS = [
    f"mlflow=={mlflow.__version__}",
    f"pandas=={pd.__version__}",
]

# Unity Catalog requires every registered model to carry a signature
# with both input and output types.
AGENT_SIGNATURE = ModelSignature(
    inputs=Schema([ColSpec("string", "question")]),
    outputs=Schema([ColSpec("string", "answer")]),
)

# Declare the foundation-model endpoint the agent queries as a model
# resource, so Model Serving injects scoped credentials for it.
AGENT_RESOURCES = [
    DatabricksServingEndpoint(
        endpoint_name="databricks-meta-llama-3-3-70b-instruct"
    )
]

registry_client = MlflowClient()

def wait_for_new_version(expected_total, timeout_s=180):
    """The model-version search index is eventually consistent."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        versions = registry_client.search_model_versions(
            f"name='{REGISTERED_MODEL}'"
        )
        if len(versions) >= expected_total:
            return max(int(v.version) for v in versions)
        time.sleep(5)
    raise TimeoutError("New model version did not appear in time.")

def log_agent_version(prompt_name, sha):
    """Log the deployable agent with one prompt and register it as the
    next version of claims.genai.claims_agent."""
    with mlflow.start_run(
        tags={"system_prompt_sha": sha,
              "agent_code": "r7_agent_code.py"}
    ) as run:
        mlflow.pyfunc.log_model(
            name="claims_agent",
            python_model=str(AGENT_CODE),
            artifacts={"system_prompt": str(PROMPTS_DIR / prompt_name)},
            pip_requirements=PIP_REQUIREMENTS,
            signature=AGENT_SIGNATURE,
            resources=AGENT_RESOURCES,
            registered_model_name=REGISTERED_MODEL,
        )
        return run.info.run_id

# Version 1 carries the v1 prompt.
run_v1 = log_agent_version("system_v1.md", v1_sha)
version_1 = wait_for_new_version(1)

# Version 2 carries the v2 prompt.
run_v2 = log_agent_version("system_v2.md", v2_sha)
version_2 = wait_for_new_version(2)

# Fingerprint each version with its prompt's sha.
registry_client.set_model_version_tag(
    REGISTERED_MODEL, str(version_1), "system_prompt_sha", v1_sha
)
registry_client.set_model_version_tag(
    REGISTERED_MODEL, str(version_2), "system_prompt_sha", v2_sha
)

# The v2 version serves live traffic.
registry_client.set_registered_model_alias(
    REGISTERED_MODEL, "champion", version_2
)

champion = registry_client.get_model_version_by_alias(
    REGISTERED_MODEL, "champion"
)
print("v1 -> version", version_1, "| sha", v1_sha, "| run", run_v1)
print("v2 -> version", version_2, "| sha", v2_sha, "| run", run_v2)
print("champion -> version", champion.version)

In [0]:
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

ENDPOINT_NAME = "claims-agent"

workspace_client = WorkspaceClient()

# The serving API takes a numeric version, so resolve the champion
# alias first — the endpoint therefore serves the champion version.
champion_version = registry_client.get_model_version_by_alias(
    REGISTERED_MODEL, "champion"
).version

served_entity = ServedEntityInput(
    entity_name=REGISTERED_MODEL,
    entity_version=str(champion_version),
    workload_size="Small",
    scale_to_zero_enabled=True,
)

# Create the endpoint, or roll an existing one to the champion version.
try:
    workspace_client.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            served_entities=[served_entity]
        ),
    )
except Exception as error:
    if "already exists" in str(error).lower():
        workspace_client.serving_endpoints.update_config(
            name=ENDPOINT_NAME,
            served_entities=[served_entity],
        )
    else:
        raise

# The lightweight pyfunc qualifies for express deployment, so this
# should be ready in minutes rather than an hour.
deadline = time.time() + 30 * 60
endpoint = None

while time.time() < deadline:
    endpoint = workspace_client.serving_endpoints.get(ENDPOINT_NAME)
    ready = (
        endpoint.state.ready.value
        if endpoint.state.ready else None
    )
    updating = (
        endpoint.state.config_update.value
        if endpoint.state.config_update else None
    )
    print("ready:", ready, "| config_update:", updating)
    if ready == "READY" and updating == "NOT_UPDATING":
        break
    time.sleep(30)

assert endpoint.state.ready.value == "READY", "endpoint not ready"
served = endpoint.config.served_entities[0]
print(f"{ENDPOINT_NAME} is READY, serving {served.entity_name}"
      f" version {served.entity_version} (= champion)")

In [0]:
import json
import textwrap
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

EVIDENCE_DIR = AGENTS_DIR / "evidence"
SCREENSHOTS_DIR = EVIDENCE_DIR / "screenshots"

QUESTION = (
    "What is the pre-existing disease (PED) waiting period "
    "for a corporate group plan?"
)

def query_endpoint(question, retries=5):
    """Query the live endpoint; scale-to-zero may need a cold start."""
    for attempt in range(1, retries + 1):
        try:
            return workspace_client.serving_endpoints.query(
                name=ENDPOINT_NAME,
                dataframe_records=[{"question": question}],
            )
        except Exception as error:
            print(f"attempt {attempt} failed: {error}")
            time.sleep(15)
    raise RuntimeError("endpoint query kept failing")

def extract_answer(response):
    """Unwrap the pyfunc scoring-server response envelope."""
    if hasattr(response, "predictions"):
        return response.predictions
    if isinstance(response, str):
        return response
    predictions = response.get("predictions", response)
    if isinstance(predictions, dict):
        return predictions.get("answer", str(predictions))
    if isinstance(predictions, list) and predictions:
        item = predictions[0]
        if isinstance(item, dict):
            return item.get("answer", str(item))
        return str(item)
    return str(predictions)

response = query_endpoint(QUESTION)
answer = extract_answer(response)

champion = registry_client.get_model_version_by_alias(
    REGISTERED_MODEL, "champion"
)

r7_registry = {
    "mode": "uc",
    "registered_model": REGISTERED_MODEL,
    "champion_alias": {
        "alias": "champion",
        "version": int(champion.version),
    },
    "versions": {
        str(version_1): {"system_prompt_sha": v1_sha},
        str(version_2): {"system_prompt_sha": v2_sha},
    },
    "endpoint": {
        "name": ENDPOINT_NAME,
        "served_model": REGISTERED_MODEL,
        "served_alias": "champion",
    },
    "query": {"question": QUESTION, "answer": answer},
}

EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)

with open(EVIDENCE_DIR / "r7_registry.json", "w", encoding="utf-8") as f:
    json.dump(r7_registry, f, indent=2, ensure_ascii=False)

# Evidence screenshot of the live endpoint and its answer.
figure, axis = plt.subplots(figsize=(12, 7))
axis.axis("off")

lines = [
    ("Databricks Model Serving - claims-agent", 16, True),
    ("", 10, False),
    (f"Served model: {REGISTERED_MODEL}"
     f" (champion = version {champion.version})", 12, True),
    ("Endpoint state: READY | workload: CPU Small |"
     " scale-to-zero: enabled", 12, False),
    ("Agent code: r7_agent_code.py"
     " (mlflow pyfunc, express deployment)", 12, False),
    ("", 10, False),
    ("Live query:", 12, True),
    (QUESTION, 11, False),
    ("", 10, False),
    ("Response:", 12, True),
    (str(answer), 11, False),
]

y = 0.97
for text, size, bold in lines:
    wrapped = textwrap.fill(text, width=100) if text else text
    axis.text(
        0.02, y, wrapped,
        fontsize=size,
        fontweight="bold" if bold else "normal",
        va="top",
        family="monospace",
    )
    y -= 0.05 * max(1, wrapped.count("\n") + 1) + 0.015

figure.savefig(
    SCREENSHOTS_DIR / "r7_endpoint.png",
    dpi=150, bbox_inches="tight",
)

print("answer:", answer)
print("evidence:", EVIDENCE_DIR / "r7_registry.json")
print("screenshot:", SCREENSHOTS_DIR / "r7_endpoint.png")